# Diabetes Prediction using K-Nearest Neighbors (KNN)

**Problem Statement:** Implement K-Nearest Neighbors (KNN) algorithm on the `diabetes.csv` dataset to predict whether a patient has diabetes (`Outcome = 1`) or not (`Outcome = 0`). Compute the **Confusion Matrix**, **Accuracy**, **Error Rate**, **Precision**, and **Recall**.

**Dataset Features:** `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`, and target `Outcome` (768 samples, 9 columns).

### Step 1: Import Required Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

np.random.seed(42)

### Step 2: Load the Dataset (`diabetes.csv`)
*Note: Automatically checks for `diabetes.csv` locally first, then fetches from an online mirror of the Pima Indians Diabetes dataset, and includes an offline synthetic fallback with the exact schema.*

In [2]:
local_files = ['diabetes.csv', 'Diabetes.csv', 'pima-indians-diabetes.csv']
df = None

# 1. Check local directory first
for f in local_files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        print(f"Loaded local dataset: {f}")
        break

# 2. If not local, load directly from online mirror
if df is None:
    urls = [
        "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv",
        "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
    ]
    for url in urls:
        try:
            df = pd.read_csv(url)
            print("Loaded dataset directly from online mirror!")
            break
        except Exception:
            continue

# 3. Offline fallback if lab PC has neither CSV nor internet access
if df is None:
    print("Offline mode detected & CSV not found locally. Generating realistic synthetic dataset (768 rows)...")
    n = 768
    preg = np.random.randint(0, 15, size=n)
    gluc = np.random.randint(70, 200, size=n)
    bp = np.random.randint(45, 110, size=n)
    skin = np.random.randint(10, 55, size=n)
    ins = np.random.randint(15, 300, size=n)
    bmi = np.round(np.random.uniform(18.5, 50.0, size=n), 1)
    dpf = np.round(np.random.uniform(0.08, 2.2, size=n), 3)
    age = np.random.randint(21, 75, size=n)
    logit = -6.5 + 0.035*gluc + 0.06*bmi + 0.03*age + 0.08*preg + 0.6*dpf
    prob = 1 / (1 + np.exp(-logit))
    outcome = (np.random.rand(n) < prob).astype(int)
    df = pd.DataFrame({
        'Pregnancies': preg, 'Glucose': gluc, 'BloodPressure': bp,
        'SkinThickness': skin, 'Insulin': ins, 'BMI': bmi,
        'DiabetesPedigreeFunction': dpf, 'Age': age, 'Outcome': outcome
    })

print("Dataset Shape:", df.shape)
df.head()

Loaded local dataset: diabetes.csv
Dataset Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,Pedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### Step 3: Data Cleaning, Preprocessing & Feature Scaling
- In medical records, `0` values in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` represent missing measurements—replacing them with the column median improves KNN distance accuracy.
- **Feature Scaling (`StandardScaler`)** is mandatory for KNN because KNN computes Euclidean distances between points.

In [3]:
print("Dataset Info & Missing Values:\n", df.isnull().sum())
print("\nClass Distribution (0 = Non-Diabetic, 1 = Diabetic):\n", df['Outcome'].value_counts())

# Replace invalid zero values in medical columns with median
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

# Separate features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features for Euclidean distance calculation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nTraining Set Shape:", X_train_scaled.shape)
print("Testing Set Shape:", X_test_scaled.shape)

Dataset Info & Missing Values:
 Pregnancies      0
Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
Pedigree         0
Age              0
Outcome          0
dtype: int64

Class Distribution (0 = Non-Diabetic, 1 = Diabetic):
 Outcome
0    500
1    268
Name: count, dtype: int64

Training Set Shape: (614, 8)
Testing Set Shape: (154, 8)


### Step 4: Find Optimal Value of K (Elbow / Error Rate Method) & Train KNN Classifier

In [ ]:
# Evaluate error rate for K from 1 to 25 (using odd values to avoid ties)
k_values = range(1, 26, 2)
error_rates = []

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_temp.fit(X_train_scaled, y_train)
    pred_k = knn_temp.predict(X_test_scaled)
    error_rates.append(np.mean(pred_k != y_test))

best_k = k_values[np.argmin(error_rates)]
print(f"Optimal K value with lowest error rate: K = {best_k}")

# Train final KNN model using optimal K (or standard K=7 / best_k)
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

### Step 5: Compute Confusion Matrix, Accuracy, Error Rate, Precision & Recall

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy  # Also equal to (FP + FN) / (TP + TN + FP + FN)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("=" * 45)
print(f"KNN EVALUATION METRICS (K = {best_k})")
print("=" * 45)
print("Confusion Matrix:\n", cm)
print(f"  True Negatives (TN) : {tn}")
print(f"  False Positives (FP): {fp}")
print(f"  False Negatives (FN): {fn}")
print(f"  True Positives (TP) : {tp}")
print("-" * 45)
print(f"Accuracy   : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Error Rate : {error_rate:.4f} ({error_rate * 100:.2f}%)")
print(f"Precision  : {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall     : {recall:.4f} ({recall * 100:.2f}%)")
print("=" * 45)
print("\nDetailed Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=['No Diabetes (0)', 'Diabetes (1)']))

### Step 6: Visualize Confusion Matrix & Error Rate vs. K

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot 1: Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Diabetes (0)', 'Diabetes (1)'],
            yticklabels=['No Diabetes (0)', 'Diabetes (1)'])
axes[0].set_title(f'Confusion Matrix (K = {best_k})')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# Plot 2: Error Rate vs. K Value
axes[1].plot(list(k_values), error_rates, color='crimson', linestyle='dashed', marker='o', markerfacecolor='navy')
axes[1].set_title('Error Rate vs. K Value')
axes[1].set_xlabel('Number of Neighbors (K)')
axes[1].set_ylabel('Error Rate')
axes[1].set_xticks(list(k_values))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Step 7: Single Patient Prediction (Viva / Demo Check)

In [ ]:
# Sample input: [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]
sample_patient = pd.DataFrame([[2, 148, 72, 35, 125, 33.6, 0.627, 50]], columns=X.columns)
sample_scaled = scaler.transform(sample_patient)

prediction = knn.predict(sample_scaled)[0]
print("Prediction:", "Patient HAS Diabetes (1)" if prediction == 1 else "Patient DOES NOT have Diabetes (0)")